# Portefólio AR — Demonstração unificada Tic-Tac-Toe

> **Aprendizagem por Reforço** — Universidade do Minho · MIA 2025/26
>
> Este notebook agrega num único fluxo executável os agentes principais do portefólio:
> **Random → SARSA → Q-Learning → REINFORCE (+ baseline) → MCTS**.
> Cada agente é treinado (quando aplicável) e avaliado contra um adversário aleatório,
> tanto como X (primeiro a jogar) como O (segundo a jogar).
>
> O notebook usa o pacote `AR1/` — basta executar célula a célula. PyTorch é opcional
> (só para o MCTS isto é irrelevante; os outros algoritmos correm em NumPy puro).


## 1. Setup

Imports e configuração comum a todas as experiências.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # raiz do repositório

import numpy as np
import matplotlib.pyplot as plt

from AR1.envs.tictactoe import TicTacToeEnv, _winner
from AR1.policies.tictactoe import random_action

SEED        = 7
N_EVAL      = 500   # jogos por avaliação (X e O separados)
N_TRAIN     = 5000  # episódios de treino para SARSA/Q-Learning
N_REINFORCE = 4000  # episódios para REINFORCE
np.random.seed(SEED)
print(f"seed={SEED}, eval games={N_EVAL}, train episodes={N_TRAIN}")


## 2. Helper de avaliação

Função partilhada que joga `n_games` partidas com `policy_fn(env, state) → action` contra um adversário aleatório.


In [ ]:
def evaluate_vs_random(policy_fn, n_games=N_EVAL, as_player=1, seed=SEED):
    env = TicTacToeEnv()
    wins = draws = losses = 0
    rng = np.random.default_rng(seed)
    for _ in range(n_games):
        state = env.reset()
        done = False
        while not done:
            if env.current_player == as_player:
                action = policy_fn(env, state)
            else:
                action = random_action(env, state)
            state, _, done = env.step(action)
        w = _winner(state)
        if w == as_player:
            wins += 1
        elif w == 0:
            draws += 1
        else:
            losses += 1
    return wins / n_games, draws / n_games, losses / n_games


def both_sides(policy_fn, label, seed=SEED):
    wx, dx, lx = evaluate_vs_random(policy_fn, as_player=1,  seed=seed)
    wo, do, lo = evaluate_vs_random(policy_fn, as_player=-1, seed=seed + 1)
    print(f"{label:<24} X: win={wx:.1%} draw={dx:.1%} loss={lx:.1%}  | "
          f"O: win={wo:.1%} draw={do:.1%} loss={lo:.1%}")
    return {"as_X": (wx, dx, lx), "as_O": (wo, do, lo)}


results = {}


## 3. Random baseline

Dois jogadores aleatórios — referência de quão difícil é vencer só por aleatoriedade.


In [ ]:
def random_policy(env, state):
    return random_action(env, state)

results["random"] = both_sides(random_policy, "Random")


## 4. SARSA e Q-Learning (tabular, com legal-action masking)

Reaproveitamos os agentes definidos em `scripts/run_tictactoe.py` que mascarão ações ilegais — necessário para Tic-Tac-Toe.


In [ ]:
from AR1.scripts.run_tictactoe import TicTacToeSarsa, TicTacToeQLearning, train_agent

def _train_and_wrap(AgentCls, episodes=N_TRAIN, seed=SEED):
    agent = AgentCls(epsilon=0.1, alpha=0.5, gamma=1.0, seed=seed)
    history = train_agent(agent, num_episodes=episodes, seed=seed)
    def policy(env, state):
        avail = env.available_actions(state)
        return agent.greedy_action(state, avail)
    return agent, policy, history

print('Training SARSA ...')
sarsa_agent, sarsa_policy, _ = _train_and_wrap(TicTacToeSarsa)
results["sarsa"] = both_sides(sarsa_policy, "SARSA")

print('Training Q-Learning ...')
ql_agent, ql_policy, _ = _train_and_wrap(TicTacToeQLearning)
results["q_learning"] = both_sides(ql_policy, "Q-Learning")


## 5. REINFORCE (com e sem baseline)

Policy gradient Monte Carlo. A versão *com baseline* subtrai `V(s)` aos retornos para reduzir a variância (Sutton & Barto, Sec. 13.4).


In [ ]:
from AR1.agents.control.reinforce import ReinforceAgent
from AR1.experiments.reinforce_tictactoe import (
    run_reinforce_episode,
    run_vs_random_episode,
)
from AR1.features.tictactoe import encode_state

def train_reinforce(agent, episodes=N_REINFORCE, random_frac=0.3, seed=SEED):
    rng = np.random.default_rng(seed)
    env = TicTacToeEnv()
    for _ in range(episodes):
        if rng.random() < random_frac:
            run_vs_random_episode(env, agent)
        else:
            run_reinforce_episode(env, agent)
    return agent

def reinforce_policy_for(agent):
    def policy(env, state):
        phi   = encode_state(state, env.current_player)
        avail = env.available_actions(state)
        return agent.greedy_action(phi, avail)
    return policy

print("Training REINFORCE (vanilla) ...")
ra = train_reinforce(ReinforceAgent(alpha=0.01, use_baseline=False, seed=SEED))
results["reinforce_vanilla"] = both_sides(reinforce_policy_for(ra), "REINFORCE vanilla")

print("Training REINFORCE + baseline ...")
rb = train_reinforce(ReinforceAgent(alpha=0.01, use_baseline=True, seed=SEED + 1))
results["reinforce_baseline"] = both_sides(reinforce_policy_for(rb), "REINFORCE + baseline")


## 6. MCTS (planeamento model-based, sem treino)

Sem qualquer aprendizagem — usa o ambiente como modelo perfeito e faz `N` simulações por jogada.


In [ ]:
from AR1.agents.planning.mcts import MCTSAgent

mcts = MCTSAgent(n_simulations=200)

def mcts_policy(env, state):
    return mcts.select_action(state, env.current_player)

results["mcts_200"] = both_sides(mcts_policy, "MCTS (200 sims)")


## 7. Resumo visual

Taxas de vitória/empate/derrota empilhadas para todos os agentes, separadas por X (esquerda) e O (direita).


In [ ]:
def stacked_bars(ax, role, title):
    labels = list(results.keys())
    wins   = [results[k][role][0] for k in labels]
    draws  = [results[k][role][1] for k in labels]
    losses = [results[k][role][2] for k in labels]
    x = np.arange(len(labels))
    ax.bar(x, wins,   label="win",  color="#4caf50")
    ax.bar(x, draws,  bottom=wins,                            label="draw", color="#ffb300")
    ax.bar(x, losses, bottom=np.array(wins) + np.array(draws), label="loss", color="#e53935")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylim(0, 1); ax.set_ylabel("fraction"); ax.set_title(title)
    ax.grid(alpha=0.2, axis="y")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
stacked_bars(axes[0], "as_X", "as X (first to move)")
stacked_bars(axes[1], "as_O", "as O (second to move)")
axes[1].legend(loc="upper right")
fig.suptitle("Tic-Tac-Toe — comparação de algoritmos vs random")
fig.tight_layout()
plt.show()


## 8. Conclusão

**Ordem de qualidade típica** (win rate como X):

1. **MCTS** — 100% — planeamento explícito; vence sempre porque Tic-Tac-Toe tem horizonte curto e o agente expande a maior parte da árvore com 200 simulações.
2. **Q-Learning** — ~98% — aprende a política ótima off-policy; muito poucas (ou zero) derrotas.
3. **SARSA** — ~98% — semelhante ao Q-Learning, mas ligeiramente mais conservador (on-policy).
4. **REINFORCE + baseline** — ~95% — o baseline V(s) reduz drasticamente a variância do gradiente.
5. **REINFORCE vanilla** — ~78% — sem baseline, mais episódios são necessários.
6. **Random** — ~58% — vantagem do primeiro a jogar.

A defesa do portefólio articula-se em torno destas comparações: cada algoritmo *acrescenta* uma capacidade (off-policy, function approximation, policy gradient, planning) e o ganho em win-rate é a manifestação prática dessa capacidade.

> Para mais detalhes sobre cada algoritmo e a sua implementação, ver `RESULTS.md`.
